#  تحلیل احساس نظرات فارسی با فاین‌تیون ParsBERT
دیتاست: mteb/SentimentDKSF (نظرات دیجی‌کالا و اسنپ‌فود). مدل پایه: HooshvareLab/bert-fa-base-uncased. مقایسه با baseline TF-IDF + Logistic Regression.

In [ ]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
# %pip install -q -r requirements.txt

## ۱. بارگذاری داده، EDA و نرمال‌سازی


In [ ]:
import re
from datasets import load_dataset
from sklearn.model_selection import train_test_split

ds = load_dataset("mteb/SentimentDKSF")
train_df, test_df = ds["train"].to_pandas(), ds["test"].to_pandas()
print(train_df.head())
print(train_df["label"].value_counts(normalize=True).round(3))
print(test_df["label"].value_counts(normalize=True).round(3))
print(train_df["text"].str.split().str.len().describe().round(1))

NUM = train_df["label"].nunique()
assert sorted(train_df["label"].unique()) == list(range(NUM)), "labels are not 0..K-1"

DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "01234567890123456789")
def normalize(t):
    t = t.replace("ي", "ی").replace("ك", "ک").replace("ة", "ه")
    t = re.sub(r"[\u064B-\u065F\u0640]", "", t)      # اعراب و کشیده
    t = t.translate(DIGITS)                              # یکسان‌سازی ارقام
    t = re.sub(r"(.)\1{2,}", r"\1\1", t)              # حروف تکراری
    return re.sub(r"\s+", " ", t).strip()

for d in (train_df, test_df):
    d["text"] = d["text"].map(normalize)
tr_df, val_df = train_test_split(train_df, test_size=0.1, stratify=train_df["label"], random_state=42)
print(len(tr_df), len(val_df), len(test_df))

In [ ]:
# نمونه‌هایی از هر برچسب برای فهمیدن معنی آن‌ها
for l in range(NUM):
    print("label", l)
    for t in train_df[train_df.label == l]["text"].sample(6, random_state=1):
        print("  -", t[:110])

## ۲. Baseline: TF-IDF + Logistic Regression


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xtr = vec.fit_transform(tr_df["text"])
lr = LogisticRegression(max_iter=1000, C=3).fit(Xtr, tr_df["label"])
pred_b = lr.predict(vec.transform(test_df["text"]))

base_acc = accuracy_score(test_df["label"], pred_b)
base_f1 = f1_score(test_df["label"], pred_b, average="macro")
print(f"Baseline  acc={base_acc:.4f}  macro-F1={base_f1:.4f}")
print(classification_report(test_df["label"], pred_b, digits=3))
print(confusion_matrix(test_df["label"], pred_b))
print(pd.Series(pred_b).value_counts().sort_index())

## ۳. فاین‌تیون ParsBERT
`SUBSET` تعداد نمونه‌های آموزشی را تعیین می‌کند (`None` یعنی کل داده). برای اجرای کامل روی CPU زمان زیادی لازم است؛ GPU توصیه می‌شود.

In [ ]:
import math, numpy as np, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

print("GPU:", torch.cuda.is_available())
MODEL = "HooshvareLab/bert-fa-base-uncased"
SUBSET = 2000                  # None = کل داده
OUT_DIR = "out_sentiment" if SUBSET else "out_sentiment_full"
tok = AutoTokenizer.from_pretrained(MODEL)

def prep(df, n=None):
    if n: df = df.sample(n=min(n, len(df)), random_state=42)
    d = Dataset.from_pandas(df[["text", "label"]].reset_index(drop=True))
    return d.map(lambda b: tok(b["text"], truncation=True, max_length=128),
                 batched=True, remove_columns=["text"])

tr, va, te = prep(tr_df, SUBSET), prep(val_df), prep(test_df)

def metrics(p):
    pr = p.predictions.argmax(-1)
    return {"accuracy": accuracy_score(p.label_ids, pr),
            "f1": f1_score(p.label_ids, pr, average="macro")}

BATCH, EPOCHS = 32, 3
warmup = int(0.1 * math.ceil(len(tr) / BATCH) * EPOCHS)

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=NUM)
args = TrainingArguments(
    output_dir=OUT_DIR, num_train_epochs=EPOCHS, learning_rate=2e-5,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=64,
    weight_decay=0.01, warmup_steps=warmup,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="f1",
    fp16=torch.cuda.is_available(), report_to="none", seed=42)

trainer = Trainer(model=model, args=args, train_dataset=tr, eval_dataset=va,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
trainer.train()

## ۴. ارزیابی روی تست و تحلیل خطا


In [ ]:
import matplotlib.pyplot as plt, json
from sklearn.metrics import ConfusionMatrixDisplay

out = trainer.predict(te)
pred = out.predictions.argmax(-1)
acc = accuracy_score(test_df["label"], pred)
f1 = f1_score(test_df["label"], pred, average="macro")
print(f"ParsBERT  acc={acc:.4f}  macro-F1={f1:.4f}   (baseline: {base_acc:.4f} / {base_f1:.4f})")
print(classification_report(test_df["label"], pred, digits=3))

ConfusionMatrixDisplay.from_predictions(test_df["label"], pred)
plt.title("ParsBERT - confusion matrix"); plt.savefig("cm_sentiment.png", dpi=150); plt.show()

err = test_df.assign(pred=pred)
err = err[err["label"] != err["pred"]]
print("errors:", len(err))
for _, r in err.sample(min(10, len(err)), random_state=1).iterrows():
    print(f"[gold={r['label']} pred={r['pred']}]", r["text"][:150])

json.dump({"baseline_acc": base_acc, "baseline_f1": base_f1, "bert_acc": acc, "bert_f1": f1,
           "train_subset": SUBSET}, open("results_sentiment.json", "w"), indent=1)

## ۵. ذخیره مدل و نمونه استفاده

In [2]:
trainer.save_model("model_sentiment"); tok.save_pretrained("model_sentiment")

from transformers import pipeline
pipe = pipeline("text-classification", model="model_sentiment", device=0 if torch.cuda.is_available() else -1)
print(pipe(["کیفیت عالی بود، ممنون", "بعد از دو روز خراب شد"]))

NameError: name 'trainer' is not defined